# 04c - Reranker Evaluation

Evaluates a cross-encoder reranker on top-30 retrieval candidates. The default candidate generator is dense retrieval because it performed best on article-level metrics in the first ablation.

In [ ]:
from pathlib import Path
import json
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running in Google Colab; using local filesystem paths.')

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path.insert(0, str(DRIVE_ROOT))

config = json.loads((DRIVE_ROOT / 'project_config.json').read_text(encoding='utf-8'))
benchmark_csv = DRIVE_ROOT / config['benchmark_csv']
index_root = DRIVE_ROOT / config.get('official_index_root', 'indexes/official_law_v3')
output_dir = DRIVE_ROOT / 'outputs/retrieval_eval'

for path in [benchmark_csv, index_root / 'index_manifest.json']:
    if not path.exists():
        raise FileNotFoundError(path)

benchmark_csv, index_root, output_dir

In [ ]:
import importlib.util
import subprocess
import sys

required_modules = {
    'sentence_transformers': 'sentence-transformers',
    'faiss': 'faiss-cpu',
    'rank_bm25': 'rank-bm25',
    'tqdm': 'tqdm',
}

missing_packages = [package for module, package in required_modules.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('All reranker dependencies are already installed.')

In [ ]:
import torch
from src.evaluation_reranker import evaluate_reranker

device = 'cuda' if torch.cuda.is_available() else 'cpu'
reranker_model = 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'

summary = evaluate_reranker(
    benchmark_csv=benchmark_csv,
    index_root=index_root,
    output_predictions_csv=output_dir / 'dense_top30_reranker_predictions_v1.csv',
    output_summary_json=output_dir / 'dense_top30_reranker_summary_v1.json',
    candidate_mode='dense',
    candidate_k=30,
    top_k=10,
    reranker_model=reranker_model,
    batch_size=16 if device == 'cuda' else 4,
    device=device,
)

summary

In [ ]:
import pandas as pd

baseline = pd.read_csv(output_dir / 'retrieval_weight_sweep_summary_v1.csv', dtype=str, keep_default_na=False)
reranker_row = {'mode': 'dense_top30+reranker', 'question_count': summary['question_count']}
reranker_row.update(summary['metrics'])

compare_cols = [
    'mode', 'question_count', 'doc_hit@5', 'doc_hit@10',
    'article_hit@5', 'article_hit@10', 'article_mrr',
    'article_ndcg@5', 'article_ndcg@10',
]
comparison = pd.concat([
    baseline[baseline['mode'].isin(['dense', 'bm25'])][compare_cols],
    pd.DataFrame([reranker_row])[compare_cols],
], ignore_index=True)
comparison_path = output_dir / 'retrieval_reranker_comparison_v1.csv'
comparison.to_csv(comparison_path, index=False, encoding='utf-8-sig')
comparison

Expected outputs:

- `outputs/retrieval_eval/dense_top30_reranker_predictions_v1.csv`
- `outputs/retrieval_eval/dense_top30_reranker_summary_v1.json`
- `outputs/retrieval_eval/retrieval_reranker_comparison_v1.csv`

If reranking improves article-level top-5 metrics, use this retriever setup for Base RAG and Fine-tuned RAG generation.